# A0.1 · How a lesson runs — skills, one shared runtime, and Kaggle

**Function A — Securing AI Architectures → How This Commons Runs**  ·  *Both directions*

| | |
|---|---|
| Tools used | standard library only |

## What this lesson is

**What it covers.** The mechanism every other lesson uses: an agent skill as a
file, one shared runtime that parses and executes it, and how both reach a
Kaggle kernel.

**Why a security engineer needs it.** A procedure you can only run inside
somebody else's curriculum is a demo. These are files — a `SKILL.md` a coding
agent loads, a script it calls, and a runtime you can point at your own
notebooks. This lesson is how you take them with you.

## 1 · The hook

Every lesson here ends in a cell you press Run on, and for 116 of them the cell does the same three things. Not knowing which three is the difference between learning a procedure and learning to press Run. It is also the difference between using these skills here and using them in a notebook of your own, which is the point of writing them down as files at all.

> **At CyberTravels.** Every procedure in this commons is a file you can copy into your own agent, and CyberTravels is what they are all demonstrated against. This lesson is the exception that explains the rule: its subject is the mechanism, so the system it runs against is the commons itself.

## 2 · The framework

```
   the two cells every lesson ends in

   +---------------------------------------------+
   | 1  SKILL.md, verbatim                        |   the procedure
   +---------------------------------------------+
   | 2  load the shared runtime, then run it      |   the machinery
   +---------------------------------------------+
                    |
        +-----------+-----------+
        |                       |
   on Kaggle:              in a checkout:
   a kernel attached       skills/_runtime/
   as a source             on disk

   one runtime. not one per notebook.
```

Every lesson in this commons ends the same way, and it is worth knowing how
before you press Run on a hundred of them.

**A skill is a file, not a topic.** `SKILL.md` carries YAML frontmatter — a
name, a description, the tools it is allowed — and a markdown body: when to use
it, the procedure, an output contract, and the ways it goes wrong. A coding
agent loads that file directly. So can you: copy the directory into
`~/.claude/skills/` and the procedure is available outside this repository,
which is the whole reason for writing it as a file.

**The frontmatter and the body are for different moments.** The description is
what an agent *routes* on — it decides whether this skill fires at all — and a
vague one means the skill never fires when it should. The body is what gets
followed once it has. `scripts/check_skills.py` fails the build when two
descriptions score identically on a plausible task, because then the winner is
whichever sorted first.

**One runtime, loaded rather than copied.** Parsing a SKILL.md, reading its
contract and calling a model are the same in every lesson. They used to be
copied into every notebook — 9,730 lines of identical code, fixable only by
rebuilding all of them. They now live once in
`skills/_runtime/cyber_commons_skill_runtime.py`.

Getting that onto Kaggle needs two facts that are not in the documentation:

- a kernel attached through `kernelDataSources` is mounted as `__script__.py`
  and is **not** on `sys.path`, so it is loaded by path rather than imported;
- the mount appears at `/kaggle/input/<slug>/` on some kernels and
  `/kaggle/input/notebooks/<user>/<slug>/` on others. Both occur; match either.

**The contract is checkable and it has a ceiling.** It says what the output must
look like, which is what makes a skill something you can hold to account. It
cannot say whether the output is *true*: a vacuous result with every field
present and every enumeration satisfied conforms perfectly. The script below
demonstrates that rather than asserting it, which is why every skill here
carries failure modes as well as a contract.

## 3 · Wiring it into a notebook of your own

On Kaggle, add the runtime as a source — *File → Add data → Notebooks*, or `kernelDataSources` if you push through the API — and the loader below finds it. In a checkout, `skills/_runtime/` is already on the path. Nothing else is needed: no install, no internet, standard library only.

## 4 · The skill, and the runtime running it on itself

In [ ]:
# skills/commons/how-a-lesson-runs/SKILL.md — embedded verbatim from the repository.
# This is the file itself, not a paraphrase of it.
SKILL_MD = r"""---
name: how-a-lesson-runs
description: >-
  Load an agent skill and execute it the way every lesson in this commons does —
  read the SKILL.md, split its frontmatter from its procedure, check its output
  contract, and run its script — using one shared runtime rather than a copy per
  notebook. Use to learn the mechanism, or to wire the same runtime into your own
  notebooks.
allowed-tools: Read, Grep, Glob
---

# How a lesson runs

Every lesson here does the same three things, in the same order, and this skill
is that procedure applied to itself. Reading it is the fastest way to understand
what the other 116 lessons are doing, and to do it in a notebook of your own.

## When to use this

First — before any other lesson, if you want to know what you are looking at.
Again later, when you want to run these skills somewhere that is not this
repository.

## Procedure

**1 — Put the SKILL.md first.** The procedure is the thing being taught, so it
is the first cell. A notebook that opens on sixty lines of parser has put the
machinery above the point, and a reader who scrolls past machinery to reach the
content learns to scroll past it everywhere.

**2 — Load the shared runtime rather than carrying it.** The parser, the
contract checker and the model adapter are the same in every lesson. Carried
per notebook they were 9,730 lines of identical code that could only be fixed by
rebuilding everything. Loaded, they are one file with one place to fix.

On Kaggle the runtime is attached to the kernel as a **source** — a script
kernel, listed in `kernelDataSources`. Two things about that are worth knowing
before you rely on it, because neither is in the documentation and both cost an
afternoon:

- the attached script does **not** land on `sys.path`, and it is **not** named
  after the kernel — it is mounted as `__script__.py`, so it is loaded by path;
- the mount has two layouts, `/kaggle/input/<slug>/` and
  `/kaggle/input/notebooks/<user>/<slug>/`, and both occur. Match either.

**3 — Split the frontmatter from the body.** The frontmatter is what an agent
*routes* on — the description decides whether this skill fires at all — and the
body is what it follows once it does. They are different audiences for different
purposes, and a skill that blurs them fires at the wrong time.

**4 — Read the output contract, and check something against it.** The contract
is the JSON block under `## Output contract`. It is what makes a skill checkable
instead of aspirational — and it has a ceiling worth stating out loud: **an
empty result conforms perfectly.** Conformance is a statement about the
serialiser. Accuracy costs more.

**5 — Run the skill's own script.** Not a paraphrase of it: the file in
`skills/<area>/<name>/scripts/`. A lesson that reimplements what a skill does
teaches the reimplementation, and the two drift the first time either changes.

## Output contract

```json
{
  "skill": {"name": "str", "description_words": 0, "allowed_tools": ["str"], "procedure_lines": 0},
  "runtime": {"loaded_from": "kaggle|repository", "path": "str", "shared": true},
  "contract": {"keys": ["str"], "conforms": true, "conformance_proves_accuracy": false},
  "script": {"path": "str", "ran": true}
}
```

## Failure modes

- **Copying the runtime into each notebook.** It works, and then a fix means
  rebuilding all of them and hoping.
- **Reading conformance as correctness.** The empty result passes.
- **Importing the attached kernel by name.** It is not on the path and it is not
  called what you think; load it by path.
"""

In [ ]:
# Execute the skill above, using the shared runtime rather than a copy.
import glob, importlib.util, os, sys

# Kaggle mounts an attached kernel under /kaggle/input, and it uses two
# different layouts — /kaggle/input/<slug>/ on some kernels and
# /kaggle/input/notebooks/<user>/<slug>/ on others. Both were observed on the
# same account in the same hour, so match either. The recursive glob is cheap
# here because /kaggle/input holds only what is attached; globbing the working
# tree instead cost eleven seconds a notebook.
_WHERE = (sorted(glob.glob("/kaggle/input/**/cyber-commons-skill-runtime/__script__.py",
                           recursive=True))
          + [os.path.join(p, "skills/_runtime/cyber_commons_skill_runtime.py")
             for p in (".", "..", "../..")])
_found = next((p for p in _WHERE if os.path.isfile(p)), None)
if _found is None:
    # Say what was looked for and what is actually there. "The runtime is
    # missing" on its own costs whoever hits it an afternoon.
    raise SystemExit("The shared skill runtime is missing."
                     "  looked at: " + repr(_WHERE) +
                     "  /kaggle/input holds: " +
                     repr(glob.glob("/kaggle/input/**", recursive=True)[:20]) +
                     "  cwd: " + os.getcwd() +
                     ". On Kaggle it is attached to this notebook as a "
                     "source; locally it is skills/_runtime/ in the repository.")
_spec = importlib.util.spec_from_file_location("cyber_commons_skill_runtime", _found)
cyber_commons_skill_runtime = importlib.util.module_from_spec(_spec)
sys.modules["cyber_commons_skill_runtime"] = cyber_commons_skill_runtime
_spec.loader.exec_module(cyber_commons_skill_runtime)
from cyber_commons_skill_runtime import run_skill

# Split skills/commons/how-a-lesson-runs/SKILL.md into the two halves an agent uses —
# the frontmatter it routes on, and the body it follows.
meta, body = run_skill(SKILL_MD)

In [ ]:
# skills/commons/how-a-lesson-runs/scripts/how_a_lesson_runs.py — embedded verbatim from the repository.
# This is the skill's own script, not a paraphrase of it.
#!/usr/bin/env python3
"""Execute this skill with the shared runtime, on itself.

The other 116 lessons load the runtime and run a skill. This one loads the
runtime and runs *this* skill, so the mechanism is visible rather than
described: where the runtime came from, what the frontmatter says, what the
contract requires, and what conformance does and does not prove.

Standard library only, and deterministic, so it runs on a Kaggle kernel with
the internet switched off.
"""

# The shared runtime, found the way every lesson finds it. In a notebook the
# cell above has already loaded it; standalone, look in the same two places.
import glob as _glob, importlib.util as _ilu, os as _os, sys as _sys

_KAGGLE = "/kaggle/input/**/cyber-commons-skill-runtime/__script__.py"
_REPO = "skills/_runtime/cyber_commons_skill_runtime.py"

if "cyber_commons_skill_runtime" not in _sys.modules:
    _where = (sorted(_glob.glob(_KAGGLE, recursive=True))
              + [_os.path.join(p, _REPO) for p in (".", "..", "../..",
                 _os.path.join(_os.path.dirname(__file__), "../../../_runtime"))])
    _found = next((p for p in _where if _os.path.isfile(p)), None)
    if _found is None:
        raise SystemExit("shared skill runtime not found; looked at " + repr(_where))
    _spec = _ilu.spec_from_file_location("cyber_commons_skill_runtime", _found)
    _mod = _ilu.module_from_spec(_spec)
    _sys.modules["cyber_commons_skill_runtime"] = _mod
    _spec.loader.exec_module(_mod)

_runtime = _sys.modules["cyber_commons_skill_runtime"]
from cyber_commons_skill_runtime import check, contract_of, parse_skill

import pathlib

# Deliberately not the path. It is /kaggle/input/... on Kaggle and a repository
# path locally, and `kaggle_verify.py` compares this notebook's remote output
# against a local run line for line — so a lesson that prints where it is
# running can never be verified. Print the property instead of the location.
print("1 · the runtime")
print("   loaded  : one shared file — the kernel attached as a source on")
print("             Kaggle, skills/_runtime/ in a local checkout")
print("   copies  : none. 9,730 lines of it used to be in the notebooks")
print()

# ---------------------------------------------------------------- the skill
SKILL_MD = globals().get("SKILL_MD") or (
    pathlib.Path(__file__).resolve().parent.parent / "SKILL.md").read_text()

meta, body = parse_skill(SKILL_MD)
print("2 · the frontmatter — what an agent routes on")
print(f"   name        : {meta['name']}")
print(f"   description : {len(meta['description'].split())} words")
print(f"   allowed     : {', '.join(meta.get('allowed-tools', [])) or '-'}")
print(f"3 · the body — what it follows once it fires: {len(body.splitlines())} lines")
print()

# ------------------------------------------------------------- the contract
contract = contract_of(body)
print("4 · the output contract")
print(f"   keys        : {', '.join(sorted(contract))}")

INSTANCE = {
    "skill": {"name": meta["name"],
              "description_words": len(meta["description"].split()),
              "allowed_tools": meta.get("allowed-tools", []),
              "procedure_lines": len(body.splitlines())},
    "runtime": {"loaded_from": "kaggle", "path": "<attached kernel or checkout>",
                "shared": True},
    "contract": {"keys": sorted(contract), "conforms": True,
                 "conformance_proves_accuracy": False},
    "script": {"path": "skills/commons/how-a-lesson-runs/scripts/how_a_lesson_runs.py",
               "ran": True},
}
problems = check(INSTANCE, contract)
print(f"   conforms    : {not problems}"
      + (f"  ({len(problems)} problem(s))" if problems else ""))

# The ceiling, demonstrated rather than asserted. This instance is the right
# shape, satisfies every type and every enumeration, and says nothing.
VACUOUS = {"skill": {"name": "", "description_words": 0, "allowed_tools": [],
                     "procedure_lines": 0},
           "runtime": {"loaded_from": "repository", "path": "", "shared": False},
           "contract": {"keys": [], "conforms": False,
                        "conformance_proves_accuracy": False},
           "script": {"path": "", "ran": False}}
print(f"   a vacuous result conforms too : {not check(VACUOUS, contract)}")

# And the one thing the contract does catch: a value outside an enumeration.
NONSENSE = {**VACUOUS, "runtime": {**VACUOUS["runtime"], "loaded_from": ""}}
print(f"   a value outside an enum is caught : {bool(check(NONSENSE, contract))}")
print()
print("So a contract checks shape, types and enumerations, and nothing else.")
print("The vacuous instance above passes every one of those and is empty - which")
print("is why every skill here carries failure modes as well as a contract.")

assert not problems, problems
assert not check(VACUOUS, contract), "a vacuous instance must conform - that is the point"
assert check(NONSENSE, contract), "an out-of-enum value must be caught"

## What you just proved

The runtime reports that it was loaded once and shared rather than copied. The frontmatter splits from the body — 59 words of routing description, three allowed tools, a 67-line procedure. An instance built from this skill conforms to its own contract, and so does a vacuous one with every field empty; a value outside an enumeration is the one thing caught. Shape, types and enumerations is all a contract checks.

## Your turn

Copy `skills/commons/how-a-lesson-runs/` into `~/.claude/skills/` and ask your own coding agent to follow it against a skill you wrote. If it cannot tell from the description alone whether the skill applies, the description is the thing to fix first.

## Where this leaves you

**What you can do now.** You can load an agent skill, split its frontmatter from its procedure, check an instance against its output contract, and run the whole thing from a Kaggle kernel with one shared runtime rather than a copy per notebook.

**What you still cannot do.** You can run a procedure and you cannot yet say which procedure this system needs. Nothing so far names a component, a risk, or a control — the mechanism is neutral about what you point it at, which is exactly why it cannot tell you where to start.

**Function A begins with the system itself. Open A1.0.**

---

**Next → [A1.0 · Start here — what securing an AI architecture means](https://spbreed.github.io/cyber-commons/lessons/A1.0.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A0.1.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A0.1.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*